## tl;dr
연도별로 과거 자료에서 월과 종료일을 선택하는 기준모형의 재현 검사입니다. 기존 선취매 전체 모델·실제 체결·기업행위 검증 결과가 아닙니다. 원시 가격 진단만 산출하며 verified_count는 0입니다.
실행 환경: 저장소 .venv의 pandas/numpy/pyarrow. API 키나 네트워크 요청은 사용하지 않습니다.

## Context & Methods
각 테스트 연도 1월 1일에 직전 5년의 완료 월을 비교합니다. 최소 3개년 표본, 월간 시가→종가 변화 중앙값 최대 월을 선택합니다. 학습 표본 피크일 중앙값(최대 28일)을 종료일로 고정합니다. 검증 연도 첫 월 시가→고정 종료일 이후 첫 관측 거래일 종가입니다. 현재 연도는 제외합니다.
### Key Assumptions
현재 종목의 사후 수집 시세이며 PIT 유니버스·상장폐지·기업행위·공식 거래일·다중검정은 미검증입니다. 진입 불가·누락 표본을 전체 연도 수에서 삭제하지 않습니다.

In [ ]:
from pathlib import Path
from datetime import date
import sys, json
import pandas as pd
root = Path('C:/Users/a4jud/kr_quant_research')
sys.path.insert(0, str(root / 'src'))
from kr_quant.research.season_holdout import evaluate_season_holdout
from kr_quant.hashing import sha256_file
cutoff = date(2026, 9, 7)
tickers = ['005930', '267260', '088130']


## Data
로컬 data/staged/live/prices.parquet 중 3개 종목을 점검합니다. 이 표본은 전 종목 대표 표본이 아니며 결과 확인을 위한 사례입니다. 원천 해시를 기록합니다.

In [ ]:
source = root / 'data/staged/live/prices.parquet'
prices = pd.read_parquet(source, columns=['ticker', 'trade_date', 'open', 'close', 'volume'], filters=[('ticker', 'in', tickers)])
prices['trade_date'] = pd.to_datetime(prices['trade_date'])
sessions = pd.read_parquet(source, columns=['trade_date']).trade_date.drop_duplicates()
assert not prices.duplicated(['ticker', 'trade_date']).any()
print({'source': str(source), 'sha256': sha256_file(source), 'rows': len(prices), 'min_date': str(prices.trade_date.min()), 'max_date': str(prices.trade_date.max())})


## Results
아래 평균·상승 비율은 유효 원시 가격 진단 표본에 한하며 연환산 수익률이나 미래 승률이 아닙니다.

In [ ]:
reports = {ticker: evaluate_season_holdout(prices, ticker=ticker, as_of=cutoff, sessions=sessions, lookback_years=5) for ticker in tickers}
summary = pd.DataFrame([{'ticker': ticker, **report['summary']} for ticker, report in reports.items()])
print(summary[['ticker', 'folds_total', 'diagnostic_count', 'verified_count', 'mean', 'median', 'positive_fraction']].to_string(index=False))
assert all(report['summary']['verified_count'] == 0 for report in reports.values())


In [ ]:
for ticker, report in reports.items():
    series = prices[prices.ticker == ticker].set_index('trade_date')
    for fold in report['folds']:
        if fold['diagnostic_return'] is None:
            continue
        entry = float(series.loc[pd.Timestamp(fold['entry_date']), 'open'])
        exit_price = float(series.loc[pd.Timestamp(fold['exit_date']), 'close'])
        assert abs((exit_price / entry - 1) - fold['diagnostic_return']) < 1e-7
        assert max(fold['selection']['train_years']) < fold['test_year']
print('Independent arithmetic and training-year boundary checks passed')


In [ ]:
baseline = next(f for f in reports['005930']['folds'] if f['test_year'] == 2023)
changed = prices.copy()
changed.loc[changed.trade_date > pd.Timestamp(baseline['exit_date']), ['open', 'close']] = 99999
after = evaluate_season_holdout(changed, ticker='005930', as_of=cutoff, sessions=sessions, lookback_years=5)
assert next(f for f in after['folds'] if f['test_year'] == 2023) == baseline
print('Post-exit future-price mutation does not change the 2023 fold')


## Takeaways
수익 산식과 학습/검증 시점 분리는 검사할 수 있습니다. 그러나 이 검사는 현재 선취매 전체 알고리즘의 수익성 입증이 아닙니다. 기존 진입 30일 전, 수급, 등급, 전체 종목 순위 규칙을 별도로 재생하고 기업행위·PIT 유니버스·체결 근거를 연결해야 합니다.
환경 제한: 이 프로젝트에는 nbformat/nbclient/ipykernel이 없어 Jupyter 커널 실행은 미검증입니다. 같은 Python 코드셀의 순차 실행 결과는 동반 MD에 기록합니다.